In [ ]:
# --- apiModule.py ---
import pandas as pd
import numpy as np

from joblib import load

# Load datasets
forecast_df = pd.read_csv("scoring_system/datasets/sarimax_forecasts_2025_2050.csv")
static_df = pd.read_csv("scoring_system/datasets/static/static_features_uganda_cities_.csv")

# Load models
models = {
    "MonsoonIntensity": load_model("scoring_system/best_model_monsoon_intensity"),
    "ClimateChange": load_model("scoring_system/best_model_climate_change"),
    "Siltation": load_model("scoring_system/best_model_siltation"),
    "AgriculturalPractices": load_model("scoring_system/best_model_agricultural_practices"),
    "Landslides": load_model("scoring_system/best_model_landslide_risks")
}

def get_prediction_dataframe(city, date):
    city = city.capitalize()  # Standardize casing
    forecast_row = forecast_df[(forecast_df['city'] == city) & (forecast_df['date'] == date)]
    if forecast_row.empty:
        raise ValueError("No forecast data found for given city and date.")

    # Predict using models (rounded int output)
    predictions = {}
    for name, model in models.items():
        pred = predict_model(model, data=forecast_row)
        predictions[name] = int(round(pred.iloc[0, -1]))

    static_row = static_df[static_df['City'] == city]
    if static_row.empty:
        raise ValueError("No static data found for given city.")

    static_features = static_row.drop(columns=['City']).iloc[0].to_dict()

    final_data = {
        "MonsoonIntensity": predictions.get("MonsoonIntensity"),
        "ClimateChange": predictions.get("ClimateChange"),
        "Siltation": predictions.get("Siltation"),
        "AgriculturalPractices": predictions.get("AgriculturalPractices"),
        "Landslides": predictions.get("Landslides"),
        **static_features
    }

    final_columns = [
        "MonsoonIntensity", "TopographyDrainage", "RiverManagement", "Deforestation", "Urbanization",
        "ClimateChange", "DamsQuality", "Siltation", "AgriculturalPractices", "Encroachments",
        "IneffectiveDisasterPreparedness", "DrainageSystems", "CoastalVulnerability", "Landslides",
        "Watersheds", "DeterioratingInfrastructure", "PopulationScore", "WetlandLoss",
        "InadequatePlanning", "PoliticalFactors"
    ]

    final_df = pd.DataFrame([{col: final_data.get(col, None) for col in final_columns}])
    scaler = load('Core_system/scaler.pkl')
    res_df = scaler.transform(final_df)
    res_df = pd.DataFrame(res_df, columns=final_columns)

    # Feature Engineering
    res_df['RunoffPotential'] = (
        res_df['MonsoonIntensity'] + res_df['Urbanization'] + res_df['Deforestation'] +
        res_df['AgriculturalPractices'] + res_df['Siltation']
    ) / 5

    res_df['DrainageCapacity'] = (
        res_df['TopographyDrainage'] + res_df['RiverManagement'] +
        res_df['DrainageSystems'] + res_df['DamsQuality']
    ) / 4

    res_df['FloodSpreadPotential'] = (
        res_df['WetlandLoss'] + res_df['Encroachments'] +
        res_df['CoastalVulnerability'] + (1 - res_df['Watersheds'])
    ) / 4

    res_df['VulnerabilityIndex'] = (
        res_df['PopulationScore'] + res_df['InadequatePlanning'] +
        res_df['IneffectiveDisasterPreparedness'] + res_df['PoliticalFactors']
    ) / 4

    res_df['FloodSizeScore'] = (
        res_df['RunoffPotential'] + res_df['FloodSpreadPotential'] - res_df['DrainageCapacity']
    )

    flood_model = load('Core_system/flood_prediction_model.pkl')
    flood_prediction = flood_model.predict(res_df)

    print("Prediction type:", type(flood_prediction))
    print("Prediction content sample:", flood_prediction)
    print("Prediction shape:", flood_prediction.shape if isinstance(flood_prediction, np.ndarray) else "Not a NumPy array")

    FloodProbability, FloodSizeScore, VulnerabilityIndex = flood_prediction[0]

    result_df = pd.DataFrame([{
        "FloodProbability": FloodProbability,
        "FloodSizeScore": FloodSizeScore,
        "VulnerabilityIndex": VulnerabilityIndex
    }])


    return pd.concat([res_df, result_df], axis=1)

# Example usage:
df = get_prediction_dataframe("Kampala", "2026-01-01")
print(df.head())

### New API

In [1]:
import pandas as pd
import numpy as np
from joblib import load

def get_flood_prediction(city, month):
    """
    Generate flood predictions for a specific city and month using forecasted and static features.
    
    Parameters:
    - city (str): Name of the city (e.g., 'Kampala').
    - month (str): Date in YYYY-MM-DD format (e.g., '2026-01-01').
    
    Returns:
    - pd.DataFrame: Single row with date, city, original features, engineered features, and predictions.
    """
    # Load datasets
    try:
        forecast_df = pd.read_csv("scoring_system/city_timeseries_future_2025_2050_rescaled.csv")
        static_df = pd.read_csv("scoring_system/datasets/static/static_features_uganda_cities_rescaled.csv")
    except Exception as e:
        raise ValueError(f"Error loading datasets: {e}")

    # Verify columns
    forecast_columns = ['date', 'city', 'monsoon_intensity', 'climate_change', 'siltation', 
                        'landslide_risks']
    missing_forecast_cols = [col for col in forecast_columns if col not in forecast_df.columns]
    if missing_forecast_cols:
        raise ValueError(f"Missing columns in forecast_df: {missing_forecast_cols}")

    static_columns = ['City', 'TopographyDrainage', 'RiverManagement', 'Deforestation', 'Urbanization',
                      'DamsQuality', 'AgriculturalPractices' ,'Encroachments', 'IneffectiveDisasterPreparedness', 
                      'DrainageSystems', 'InadequatePlanning', 'PoliticalFactors',
                      'CoastalVulnerability', 'Watersheds', 'DeterioratingInfrastructure', 
                      'PopulationScore', 'WetlandLoss']
    missing_static_cols = [col for col in static_columns if col not in static_df.columns]
    if missing_static_cols:
        raise ValueError(f"Missing columns in static_df: {missing_static_cols}")

    # Standardize city names
    city = city.capitalize()
    forecast_df['city'] = forecast_df['city'].str.capitalize()
    static_df['City'] = static_df['City'].str.capitalize()

    # Convert month to datetime
    try:
        month = pd.to_datetime(month, format='%Y-%m-%d', errors='coerce')
        if month is pd.NaT:
            raise ValueError("Invalid date format. Use YYYY-MM-DD (e.g., '2026-01-01').")
    except Exception as e:
        raise ValueError(f"Error parsing month: {e}")

    # Filter forecast data for city and month
    forecast_row = forecast_df[(forecast_df['city'] == city) & 
                              (pd.to_datetime(forecast_df['date']) == month)]
    if forecast_row.empty:
        raise ValueError(f"No forecast data found for city '{city}' and month '{month}'.")

    # Filter static data for city
    static_row = static_df[static_df['City'] == city]
    if static_row.empty:
        raise ValueError(f"No static data found for city '{city}'.")

    # Merge forecast and static data
    merged_data = forecast_row.merge(static_row, left_on='city', right_on='City', how='inner')
    if merged_data.empty:
        raise ValueError(f"Failed to merge data for city '{city}' and month '{month}'.")

    # Drop redundant City column
    merged_data = merged_data.drop(columns=['City'])

    # Define final columns for the core model
    final_columns = [
        'MonsoonIntensity', 'TopographyDrainage', 'RiverManagement', 'Deforestation', 'Urbanization', 
        'ClimateChange', 'DamsQuality', 'Siltation', 'AgriculturalPractices', 'Encroachments', 
        'IneffectiveDisasterPreparedness', 'DrainageSystems', 'CoastalVulnerability', 'Landslides', 
        'Watersheds', 'DeterioratingInfrastructure', 'PopulationScore', 'InadequatePlanning', 
        'PoliticalFactors', 'WetlandLoss'
    ]

    # Rename forecast columns to match final_columns
    merged_data = merged_data.rename(columns={
        'monsoon_intensity': 'MonsoonIntensity',
        'climate_change': 'ClimateChange',
        'siltation': 'Siltation',
        'landslide_risks': 'Landslides'
    })

    # Select and order columns
    input_df = merged_data[['date', 'city'] + final_columns]
    
    # Check for missing values
    missing_values = input_df[final_columns].isna().sum()
    if missing_values.any():
        print(f"Warning: Missing values in input_df:\n{missing_values}")
        input_df[final_columns] = input_df[final_columns].fillna(input_df[final_columns].mean())

    # Load core model
    try:
        scaler = load('Core_system/scaler.pkl')
        flood_model = load('Core_system/flood_prediction_model.pkl')
    except Exception as e:
        raise ValueError(f"Error loading scaler or model: {e}")
    
    # Scale the features
    scaled_data = scaler.transform(input_df[final_columns])
    scaled_df = pd.DataFrame(scaled_data, columns=final_columns, index=input_df.index)

    # scaled_df = input_df.copy()
    scaled_input = scaled_df
    #scaled_df = scaled_df.drop(columns=['date', 'city'], errors='ignore')
    
    # Feature engineering
    scaled_df['RunoffPotential'] = (
        scaled_df['MonsoonIntensity'] + scaled_df['Urbanization'] + scaled_df['Deforestation'] +
        scaled_df['AgriculturalPractices'] + scaled_df['Siltation']
    ) / 5

    scaled_df['DrainageCapacity'] = (
        scaled_df['TopographyDrainage'] + scaled_df['RiverManagement'] +
        scaled_df['DrainageSystems'] + scaled_df['DamsQuality']
    ) / 4

    scaled_df['FloodSpreadPotential'] = (
        scaled_df['WetlandLoss'] + scaled_df['Encroachments'] +
        scaled_df['CoastalVulnerability'] + (1 - scaled_df['Watersheds'])
    ) / 4

    # Predict with core model
    flood_predictions = flood_model.predict(scaled_df)

    # Check prediction output
    if not isinstance(flood_predictions, np.ndarray) or flood_predictions.shape != (1, 3):
        raise ValueError(f"Expected flood_predictions to be a 1x3 NumPy array, got shape {flood_predictions.shape}")

    # Create output DataFrame
    output_df = scaled_input[final_columns].copy()
    output_df['FloodProbability'] = flood_predictions[0, 0]
    output_df['FloodSizeScore'] = flood_predictions[0, 1]
    output_df['VulnerabilityIndex'] = flood_predictions[0, 2]

    # Add engineered features
    output_df['RunoffPotential'] = scaled_df['RunoffPotential']
    output_df['DrainageCapacity'] = scaled_df['DrainageCapacity']
    output_df['FloodSpreadPotential'] = scaled_df['FloodSpreadPotential']

    return output_df

# Example usage
try:
    result_df = get_flood_prediction("Kasese", "2037-12-01")
    print("\nPrediction Output:")
    print(result_df.head())
except Exception as e:
    print(f"Error: {e}")


Prediction Output:
   MonsoonIntensity  TopographyDrainage  RiverManagement  Deforestation  \
0          0.062541              0.6875              0.5          0.625   

   Urbanization  ClimateChange  DamsQuality  Siltation  AgriculturalPractices  \
0        0.5625       0.456798        0.375   0.167993                 0.5625   

   Encroachments  ...  PopulationScore  InadequatePlanning  PoliticalFactors  \
0         0.6875  ...           0.4375               0.625               0.5   

   WetlandLoss  FloodProbability  FloodSizeScore  VulnerabilityIndex  \
0       0.6875           0.32528        0.302357            0.546875   

   RunoffPotential  DrainageCapacity  FloodSpreadPotential  
0         0.396107           0.53125                0.4375  

[1 rows x 26 columns]
